In [1]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from PIL import Image
import numpy as np

In [2]:
from pathlib import Path

from PIL import Image

import numpy as np

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# 1. Dataset path
dataset_path = Path(
    r"D:\good_projects\visionguard\mvtec_anomaly_detection"
)
# 2. Path to good training images
train_good_path = dataset_path / "bottle" / "train" / "good"

# 3. Get an image path
image_path = next(train_good_path.iterdir())

print("Image path:", image_path)

# 4. Open the image
image = Image.open(image_path).convert("RGB")

print("Original size:", image.size)

# 5. Resize the image
image = image.resize((224, 224))

print("New size:", image.size)

# 6. Convert PIL image → NumPy array

image_array = np.array(image)

print("After NumPy conversion:", image_array.shape)

# 7. Add batch dimension
image_array = np.expand_dims(image_array, axis=0)

print("After adding batch dimension:", image_array.shape) #The 1 means:We are passing one image in this batch.


image_array = preprocess_input(image_array)


model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


features = model.predict(image_array, verbose=0)

print("Feature shape:", features.shape)

Image path: D:\good_projects\visionguard\mvtec_anomaly_detection\bottle\train\good\000.png
Original size: (900, 900)
New size: (224, 224)
After NumPy conversion: (224, 224, 3)
After adding batch dimension: (1, 224, 224, 3)
Feature shape: (1, 7, 7, 512)


In [3]:
model.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 14,714,688 (56.13 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
embedding = features.flatten()

print("Embedding shape:", embedding.shape)

Embedding shape: (25088,)


In [5]:
def extract_embedding(image_path, model):
    image = Image.open(image_path).convert("RGB")
    image = image.resize((224, 224))

    image_array = np.array(image)
    image_array = np.expand_dims(image_array, axis=0)
    image_array = preprocess_input(image_array)

    features = model.predict(image_array, verbose=0)

    embedding = features.flatten()

    return embedding

In [6]:
image_path = next(train_good_path.iterdir())

embedding = extract_embedding(image_path, model)

print(embedding.shape)

print(embedding[:10])

(25088,)
[-0. -0. -0. -0. -0. -0. -0. -0. -0. -0.]


In [7]:
normal_embeddings = []

for image_path in train_good_path.iterdir():

    if image_path.is_file():

        embedding = extract_embedding(image_path, model)

        normal_embeddings.append(embedding)

In [8]:
normal_embeddings = np.array(normal_embeddings)
print(normal_embeddings.shape)

(209, 25088)


In [9]:
test_path = dataset_path / "bottle" / "test" / "good"

test_image = next(test_path.iterdir())

test_embedding = extract_embedding(test_image, model)

In [10]:
distances = np.linalg.norm(
    normal_embeddings - test_embedding,
    axis=1
)
print(distances.shape)

(209,)


In [11]:
anomaly_score = distances.min()

print("Anomaly score:", anomaly_score)

Anomaly score: 282.36252


In [12]:
test_root = dataset_path / "bottle" / "test"

test_categories = [
    item for item in test_root.iterdir()
    if item.is_dir()
]

for category in test_categories:
    print(category.name)


broken_large
broken_small
contamination
good


In [13]:
broken_small_path = test_root / "broken_small"

broken_image = next(broken_small_path.iterdir())

broken_embedding = extract_embedding(
    broken_image,
    model
)

In [14]:
broken_distances = np.linalg.norm(
    normal_embeddings - broken_embedding,
    axis=1
)

broken_score = broken_distances.min()

print("Broken image anomaly score:", broken_score)

Broken image anomaly score: 386.90942


In [15]:
def calculate_anomaly_score(
    embedding,
    normal_embeddings
):
    distances = np.linalg.norm(
        normal_embeddings - embedding,
        axis=1
    )

    return distances.min()

In [16]:
normal_score = calculate_anomaly_score(
    test_embedding,
    normal_embeddings
)

broken_score = calculate_anomaly_score(
    broken_embedding,
    normal_embeddings
)

print("Normal:", normal_score)
print("Broken:", broken_score)

Normal: 282.36252
Broken: 386.90942


In [17]:
def score_image(image_path, model, normal_embeddings):
    embedding = extract_embedding(image_path, model)

    score = calculate_anomaly_score(
        embedding,
        normal_embeddings
    )

    return score

In [18]:
score = score_image(
    image_path,
    model,
    normal_embeddings
)

print(score)

0.0


In [19]:
test_root = dataset_path / "bottle" / "test"

good_path = test_root / "good"
broken_small_path = test_root / "broken_small"
broken_large_path = test_root / "broken_large"
contamination_path = test_root / "contamination"

In [20]:
categories_to_test = [
    "good",
    "broken_small",
    "broken_large",
    "contamination"
]

for category in categories_to_test:

    category_path = test_root / category

    print("\n", category)

    images = list(category_path.iterdir())[:3]

    for image_path in images:

        score = score_image(
            image_path,
            model,
            normal_embeddings
        )

        print(
            image_path.name,
            "→",
            score
        )


 good
000.png → 282.36252
001.png → 214.5145
002.png → 247.80719

 broken_small
000.png → 386.90942
001.png → 441.02054
002.png → 499.06546

 broken_large
000.png → 662.20026
001.png → 733.452
002.png → 770.6284

 contamination
000.png → 520.4301
001.png → 457.12115
002.png → 412.80304


In [21]:
import pandas as pd
results = []

for category in categories_to_test:

    category_path = test_root / category

    images = list(category_path.iterdir())[:3]

    for image_path in images:

        score = score_image(
            image_path,
            model,
            normal_embeddings
        )

        results.append({
            "image": image_path.name,
            "category": category,
            "score": score
        })

In [22]:
results_df = pd.DataFrame(results)

print(results_df)

      image       category       score
0   000.png           good  282.362518
1   001.png           good  214.514496
2   002.png           good  247.807190
3   000.png   broken_small  386.909424
4   001.png   broken_small  441.020538
5   002.png   broken_small  499.065460
6   000.png   broken_large  662.200256
7   001.png   broken_large  733.452026
8   002.png   broken_large  770.628418
9   000.png  contamination  520.430115
10  001.png  contamination  457.121155
11  002.png  contamination  412.803040


In [23]:
def calculate_leave_one_out_scores(normal_embeddings):

    scores = []

    for i in range(len(normal_embeddings)):

        query = normal_embeddings[i]

        reference = np.delete(
            normal_embeddings,
            i,
            axis=0
        )

        distances = np.linalg.norm(
            reference - query,
            axis=1
        )

        score = distances.min()

        scores.append(score)

    return np.array(scores)

In [24]:
normal_train_scores = calculate_leave_one_out_scores(
    normal_embeddings
)

print(normal_train_scores.shape)

(209,)


In [25]:
print("Minimum:", normal_train_scores.min())
print("Maximum:", normal_train_scores.max())
print("Mean:", normal_train_scores.mean())
print("Median:", np.median(normal_train_scores))

Minimum: 77.68017
Maximum: 392.61298
Mean: 247.4631
Median: 247.07661


In [26]:
threshold = np.percentile(
    normal_train_scores,
    95
)

print("Threshold:", threshold)

Threshold: 307.865
